**IMPORTS**

In [ ]:
# =====================================================
# Project: Machine Learning-Based Sanitation Management
#          & Disease Outbreak Prediction using GIS (India)
# =====================================================
# Cell 1: Imports, Configuration & Environment Setup
# =====================================================

# Data Handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Time Handling
from datetime import datetime

# System Utilities
import os
import warnings
warnings.filterwarnings("ignore")

# Set Data Path
DATA_PATH = "/content/data"

print("Environment Ready ✅")
print("Data folder contains:")
print(os.listdir(DATA_PATH))


Environment Ready ✅
Data folder contains:


FileNotFoundError: [Errno 2] No such file or directory: '/content/data'

**GIS Libraries**

In [ ]:
# =====================================================
# Cell 46: Install GIS Libraries (Silent Mode)
# =====================================================

import sys

# Install silently
!{sys.executable} -m pip install geopandas folium -q

print("GIS Libraries Installed Successfully ✅")


**Load India States GeoJSON**

In [ ]:
# =====================================================
# Cell 47: Load India State Boundaries
# =====================================================

import geopandas as gpd

# Public India states GeoJSON
india_geo = gpd.read_file(
    "https://raw.githubusercontent.com/geohacker/india/master/state/india_telengana.geojson"
)

print("Geo dataset shape:", india_geo.shape)
india_geo.head()


**DATASET LOADING**

In [ ]:
# =====================================================
# Cell 2: Load & Inspect COVID Case Data
# =====================================================

# Load COVID dataset
covid_df = pd.read_csv(f"{DATA_PATH}/covid_19_india.csv")

print("Dataset Shape:", covid_df.shape)
print("\nColumn Names:\n", covid_df.columns)

print("\nPreview:")
covid_df.head()


**DATA CONVERSION & BASIC CLEANING**

In [ ]:
# =====================================================
# Cell 3: Date Conversion & Basic Cleaning
# =====================================================

# Convert Date column to datetime
covid_df['Date'] = pd.to_datetime(covid_df['Date'], dayfirst=True)

# Sort by date
covid_df = covid_df.sort_values(by='Date')

# Reset index
covid_df.reset_index(drop=True, inplace=True)

print("Date conversion successful ✅")
print("\nDate Range:")
print("From:", covid_df['Date'].min())
print("To:", covid_df['Date'].max())

print("\nMissing Values:\n")
print(covid_df.isnull().sum())


**Convert Daily Data → Monthly State-wise Aggregation**

In [ ]:
# =====================================================
# Cell 4: Monthly Aggregation (State-wise)
# =====================================================

# Extract Year-Month
covid_df['YearMonth'] = covid_df['Date'].dt.to_period('M')

# Group by State and Month
monthly_covid = covid_df.groupby(
    ['State/UnionTerritory', 'YearMonth']
).agg({
    'Confirmed': 'max',   # cumulative cases at end of month
    'Deaths': 'max',
    'Cured': 'max'
}).reset_index()

# Convert YearMonth back to timestamp
monthly_covid['YearMonth'] = monthly_covid['YearMonth'].dt.to_timestamp()

print("Monthly dataset shape:", monthly_covid.shape)
monthly_covid.head()


**Convert Cumulative → Monthly New Cases**

In [ ]:
# =====================================================
# Cell 5: Convert Cumulative to Monthly New Cases
# =====================================================

# Sort properly
monthly_covid = monthly_covid.sort_values(
    by=['State/UnionTerritory', 'YearMonth']
)

# Calculate monthly new cases
monthly_covid['MonthlyCases'] = monthly_covid.groupby(
    'State/UnionTerritory'
)['Confirmed'].diff()

# First month will have NaN (replace with Confirmed value)
monthly_covid['MonthlyCases'] = monthly_covid['MonthlyCases'].fillna(
    monthly_covid['Confirmed']
)

print("Converted to Monthly New Cases ✅")
monthly_covid.head()


**Check & Clean Anomalies in MonthlyCases**

In [ ]:
# =====================================================
# Cell 6: Sanity Check & Cleaning MonthlyCases
# =====================================================

# Check negative values
negative_cases = monthly_covid[monthly_covid['MonthlyCases'] < 0]

print("Number of negative monthly values:", len(negative_cases))

# Replace negatives with 0 (data correction adjustment)
monthly_covid['MonthlyCases'] = monthly_covid['MonthlyCases'].apply(
    lambda x: 0 if x < 0 else x
)

# Check basic statistics
print("\nBasic Statistics of MonthlyCases:")
print(monthly_covid['MonthlyCases'].describe())


**Creating Time-Lag Features (Per State)**

In [ ]:
# =====================================================
# Cell 7: Time-Lag Feature Engineering
# =====================================================

# Sort again (important before shift)
monthly_covid = monthly_covid.sort_values(
    by=['State/UnionTerritory', 'YearMonth']
)

# Create lag features per state
monthly_covid['lag_1'] = monthly_covid.groupby(
    'State/UnionTerritory'
)['MonthlyCases'].shift(1)

monthly_covid['lag_2'] = monthly_covid.groupby(
    'State/UnionTerritory'
)['MonthlyCases'].shift(2)

monthly_covid['rolling_mean_3'] = monthly_covid.groupby(
    'State/UnionTerritory'
)['MonthlyCases'].rolling(3).mean().reset_index(level=0, drop=True)

# Drop rows with NaN from lag creation
monthly_covid = monthly_covid.dropna().reset_index(drop=True)

print("Lag features created successfully ✅")
monthly_covid.head()


**Prepare ML Dataset (Feature Selection)**

In [ ]:
# =====================================================
# Cell 8: Prepare ML Features & Target
# =====================================================

# Define features and target
feature_cols = ['lag_1', 'lag_2', 'rolling_mean_3']

X = monthly_covid[feature_cols]
y = monthly_covid['MonthlyCases']

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

X.head()


**Time-Based Train-Test Split**

In [ ]:
# =====================================================
# Cell 9: Time-Based Train-Test Split
# =====================================================

# Add Year column for splitting
monthly_covid['Year'] = monthly_covid['YearMonth'].dt.year

# Split
train = monthly_covid[monthly_covid['Year'] < 2021]
test = monthly_covid[monthly_covid['Year'] >= 2021]

X_train = train[['lag_1', 'lag_2', 'rolling_mean_3']]
y_train = train['MonthlyCases']

X_test = test[['lag_1', 'lag_2', 'rolling_mean_3']]
y_test = test['MonthlyCases']

print("Training size:", X_train.shape)
print("Testing size:", X_test.shape)


**Baseline ML Models Comparison**

In [ ]:
# =====================================================
# Cell 10: Baseline Model Training & Comparison
# =====================================================

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
import numpy as np

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "SVR": SVR(kernel='rbf')
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    results[name] = {
        "MAE": mean_absolute_error(y_test, preds),
        "RMSE": np.sqrt(mean_squared_error(y_test, preds)),
        "R2": r2_score(y_test, preds),
        "preds": preds
    }

results_df = pd.DataFrame(results).T[['MAE', 'RMSE', 'R2']]
results_df


**Fix Rolling Feature (Remove Leakage)**

In [ ]:
# =====================================================
# Cell 11: Fix Rolling Mean (Prevent Data Leakage)
# =====================================================

monthly_covid = monthly_covid.sort_values(
    by=['State/UnionTerritory', 'YearMonth']
)

monthly_covid['rolling_mean_3'] = monthly_covid.groupby(
    'State/UnionTerritory'
)['MonthlyCases'].shift(1).rolling(3).mean().reset_index(level=0, drop=True)

# Drop NaNs again
monthly_covid = monthly_covid.dropna().reset_index(drop=True)

print("Rolling mean corrected ✅")
monthly_covid.head()


**Recreate Train-Test Split (After Fix)**

In [ ]:
# =====================================================
# Cell 12: Recreate Train-Test Split (After Fix)
# =====================================================

train = monthly_covid[monthly_covid['Year'] < 2021]
test = monthly_covid[monthly_covid['Year'] >= 2021]

X_train = train[['lag_1', 'lag_2', 'rolling_mean_3']]
y_train = train['MonthlyCases']

X_test = test[['lag_1', 'lag_2', 'rolling_mean_3']]
y_test = test['MonthlyCases']

print("Training size:", X_train.shape)
print("Testing size:", X_test.shape)


**Retrain Baseline Models (Leak-Free)**

In [ ]:
# =====================================================
# Cell 13: Retrain Models After Fix
# =====================================================

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
import numpy as np

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "SVR": SVR(kernel='rbf')
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    results[name] = {
        "MAE": mean_absolute_error(y_test, preds),
        "RMSE": np.sqrt(mean_squared_error(y_test, preds)),
        "R2": r2_score(y_test, preds),
        "preds": preds
    }

results_df = pd.DataFrame(results).T[['MAE', 'RMSE', 'R2']]
results_df


**Poisson Regression**

In [ ]:
# =====================================================
# Cell 14: Poisson Regression (Count-Based Model)
# =====================================================

from sklearn.linear_model import PoissonRegressor

poisson_model = PoissonRegressor(alpha=0.01, max_iter=300)
poisson_model.fit(X_train, y_train)

poisson_pred = poisson_model.predict(X_test)

poisson_mae = mean_absolute_error(y_test, poisson_pred)
poisson_rmse = np.sqrt(mean_squared_error(y_test, poisson_pred))
poisson_r2 = r2_score(y_test, poisson_pred)

print("Poisson Regression Results")
print("MAE:", poisson_mae)
print("RMSE:", poisson_rmse)
print("R²:", poisson_r2)


**Gradient Boosting Regressor**

In [ ]:
# =====================================================
# Cell 15: Gradient Boosting Regressor
# =====================================================

from sklearn.ensemble import GradientBoostingRegressor

gbr_model = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gbr_model.fit(X_train, y_train)
gbr_pred = gbr_model.predict(X_test)

gbr_mae = mean_absolute_error(y_test, gbr_pred)
gbr_rmse = np.sqrt(mean_squared_error(y_test, gbr_pred))
gbr_r2 = r2_score(y_test, gbr_pred)

print("Gradient Boosting Results")
print("MAE:", gbr_mae)
print("RMSE:", gbr_rmse)
print("R²:", gbr_r2)


**Model Comparison Table**

In [ ]:
# =====================================================
# Cell 16: Final Model Comparison Table
# =====================================================

comparison_df = pd.DataFrame({
    'Model': [
        'Linear Regression',
        'Random Forest',
        'Poisson Regression',
        'Gradient Boosting'
    ],
    'MAE': [
        results['Linear Regression']['MAE'],
        results['Random Forest']['MAE'],
        poisson_mae,
        gbr_mae
    ],
    'RMSE': [
        results['Linear Regression']['RMSE'],
        results['Random Forest']['RMSE'],
        poisson_rmse,
        gbr_rmse
    ],
    'R2': [
        results['Linear Regression']['R2'],
        results['Random Forest']['R2'],
        poisson_r2,
        gbr_r2
    ]
})

comparison_df


**Actual vs Predicted Plot (Gradient Boosting)**

---



In [ ]:
# =====================================================
# Cell 17: Actual vs Predicted (Maharashtra Example)
# =====================================================

# Select Maharashtra only
maha_test = test[test['State/UnionTerritory'] == 'Maharashtra']

X_maha = maha_test[['lag_1', 'lag_2', 'rolling_mean_3']]
y_maha = maha_test['MonthlyCases']

# Predict using Gradient Boosting
maha_pred = gbr_model.predict(X_maha)

plt.figure(figsize=(12,5))
plt.plot(maha_test['YearMonth'], y_maha.values, marker='o')
plt.plot(maha_test['YearMonth'], maha_pred, linestyle='--')
plt.xlabel("Month")
plt.ylabel("Monthly Cases")
plt.title("Actual vs Predicted Monthly Cases (Maharashtra)")
plt.xticks(rotation=45)
plt.show()


**Calculate MAPE & Accuracy %**

In [ ]:
# =====================================================
# Cell 18: Calculate MAPE & Model Accuracy
# =====================================================

# Avoid division by zero
non_zero_mask = y_test != 0

mape = np.mean(
    np.abs((y_test[non_zero_mask] - gbr_pred[non_zero_mask]) / y_test[non_zero_mask])
) * 100

accuracy_percent = 100 - mape

print("MAPE (%):", round(mape, 2))
print("Model Accuracy (%):", round(accuracy_percent, 2))


**Log Transformation of Target**

In [ ]:
# =====================================================
# Cell 19: Log Transformation of Target Variable
# =====================================================

# Create log target
monthly_covid['LogMonthlyCases'] = np.log1p(monthly_covid['MonthlyCases'])

# Re-split data
train = monthly_covid[monthly_covid['Year'] < 2021]
test = monthly_covid[monthly_covid['Year'] >= 2021]

X_train = train[['lag_1', 'lag_2', 'rolling_mean_3']]
y_train_log = train['LogMonthlyCases']

X_test = test[['lag_1', 'lag_2', 'rolling_mean_3']]
y_test_log = test['LogMonthlyCases']

print("Log transformation complete ✅")


**Train Gradient Boosting on Log Scale**

In [ ]:
# =====================================================
# Cell 20: Train Gradient Boosting (Log Scale)
# =====================================================

from sklearn.ensemble import GradientBoostingRegressor

# Train model on log target
gbr_log_model = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gbr_log_model.fit(X_train, y_train_log)

# Predict (log scale)
log_preds = gbr_log_model.predict(X_test)

# Convert back to original scale
gbr_log_pred = np.expm1(log_preds)

print("Log-scale model trained successfully ✅")


**Evaluate Log-Scale Model**

In [ ]:
# =====================================================
# Cell 21: Evaluate Log-Scale Model
# =====================================================

# True values (original scale)
y_test_original = test['MonthlyCases']

# MAE
log_mae = mean_absolute_error(y_test_original, gbr_log_pred)

# RMSE
log_rmse = np.sqrt(mean_squared_error(y_test_original, gbr_log_pred))

# R2
log_r2 = r2_score(y_test_original, gbr_log_pred)

# MAPE (avoid zeros)
non_zero_mask = y_test_original != 0
log_mape = np.mean(
    np.abs((y_test_original[non_zero_mask] - gbr_log_pred[non_zero_mask])
           / y_test_original[non_zero_mask])
) * 100

print("Log-Model MAE:", round(log_mae, 2))
print("Log-Model RMSE:", round(log_rmse, 2))
print("Log-Model R²:", round(log_r2, 3))
print("Log-Model MAPE (%):", round(log_mape, 2))


**Load & Inspect Vaccination Data**

In [ ]:
# =====================================================
# Cell 22: Load & Inspect Vaccination Data
# =====================================================

vaccine_df = pd.read_csv(f"{DATA_PATH}/covid_vaccine_statewise.csv")

print("Dataset Shape:", vaccine_df.shape)
print("\nColumns:\n", vaccine_df.columns)

vaccine_df.head()


**Clean & Convert Vaccination Data**

In [ ]:
# =====================================================
# Cell 23: Clean & Monthly Aggregate Vaccination Data
# =====================================================

# Remove India total row
vaccine_df = vaccine_df[vaccine_df['State'] != 'India']

# Convert date
vaccine_df['Updated On'] = pd.to_datetime(
    vaccine_df['Updated On'], dayfirst=True
)

# Extract YearMonth
vaccine_df['YearMonth'] = vaccine_df['Updated On'].dt.to_period('M')

# Keep only required columns
vaccine_df = vaccine_df[['State', 'YearMonth', 'Total Doses Administered']]

# Aggregate monthly (take max cumulative per month)
monthly_vaccine = vaccine_df.groupby(
    ['State', 'YearMonth']
)['Total Doses Administered'].max().reset_index()

# Convert back to timestamp
monthly_vaccine['YearMonth'] = monthly_vaccine['YearMonth'].dt.to_timestamp()

print("Monthly vaccination dataset ready ✅")
monthly_vaccine.head()


**Convert Cumulative Vaccination → Monthly New Doses**

In [ ]:
# =====================================================
# Cell 24: Convert Vaccination to Monthly New Doses
# =====================================================

# Sort properly
monthly_vaccine = monthly_vaccine.sort_values(
    by=['State', 'YearMonth']
)

# Calculate monthly vaccination
monthly_vaccine['MonthlyVaccinations'] = monthly_vaccine.groupby(
    'State'
)['Total Doses Administered'].diff()

# Replace first month NaN with cumulative value
monthly_vaccine['MonthlyVaccinations'] = monthly_vaccine[
    'MonthlyVaccinations'
].fillna(monthly_vaccine['Total Doses Administered'])

# Replace negative corrections (if any)
monthly_vaccine['MonthlyVaccinations'] = monthly_vaccine[
    'MonthlyVaccinations'
].apply(lambda x: 0 if x < 0 else x)

print("Monthly vaccination conversion complete ✅")
monthly_vaccine.head()


**Merge Vaccination with Outbreak Data**

In [ ]:
# =====================================================
# Cell 25: Merge Vaccination with Outbreak Data
# =====================================================

# Rename column to match outbreak dataset
monthly_vaccine.rename(columns={'State': 'State/UnionTerritory'}, inplace=True)

# Merge
merged_df = pd.merge(
    monthly_covid,
    monthly_vaccine[['State/UnionTerritory', 'YearMonth', 'MonthlyVaccinations']],
    on=['State/UnionTerritory', 'YearMonth'],
    how='left'
)

# Fill missing vaccination values with 0 (before vaccination rollout)
merged_df['MonthlyVaccinations'] = merged_df[
    'MonthlyVaccinations'
].fillna(0)

print("Merged dataset shape:", merged_df.shape)
merged_df.head()


**Retrain Gradient Boosting with Vaccination Feature**

In [ ]:
# =====================================================
# Cell 26: Train Gradient Boosting with Vaccination
# =====================================================

# Time-based split again
train = merged_df[merged_df['Year'] < 2021]
test = merged_df[merged_df['Year'] >= 2021]

# Features now include vaccination
feature_cols = ['lag_1', 'lag_2', 'rolling_mean_3', 'MonthlyVaccinations']

X_train = train[feature_cols]
y_train = train['MonthlyCases']

X_test = test[feature_cols]
y_test = test['MonthlyCases']

# Train model
gbr_vaccine_model = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gbr_vaccine_model.fit(X_train, y_train)

preds_vaccine = gbr_vaccine_model.predict(X_test)

# Evaluation
mae_vaccine = mean_absolute_error(y_test, preds_vaccine)
rmse_vaccine = np.sqrt(mean_squared_error(y_test, preds_vaccine))
r2_vaccine = r2_score(y_test, preds_vaccine)

print("Gradient Boosting + Vaccination Results")
print("MAE:", round(mae_vaccine, 2))
print("RMSE:", round(rmse_vaccine, 2))
print("R²:", round(r2_vaccine, 3))


**Load One Swachh Survekshan File (Structure Check)**

In [ ]:
# =====================================================
# Cell 27: Inspect Sanitation Dataset (SS2016 example)
# =====================================================

ss2016 = pd.read_csv(f"{DATA_PATH}/SS2016.csv")

print("Shape:", ss2016.shape)
print("\nColumns:\n", ss2016.columns)

ss2016.head()


**Reload All Sanitation Files Safely**

In [ ]:
# =====================================================
# Cell 29: Reload Sanitation Files with Encoding Fix
# =====================================================

files = ["SS2016.csv", "SS2017.csv", "SS2018.csv",
         "SS2019.csv", "SS2020.csv", "SS2022.csv", "SS2023.csv"]

for file in files:
    try:
        df = pd.read_csv(f"{DATA_PATH}/{file}", encoding='latin1')
        print("\n", file)
        print("Columns:", list(df.columns))
    except Exception as e:
        print("\nError in", file, ":", e)


**Extract & Standardize 2017–2019 Sanitation Data**

In [ ]:
# =====================================================
# Cell 30: Extract Sanitation Data (2017-2019)
# =====================================================

# Load files
ss2017 = pd.read_csv(f"{DATA_PATH}/SS2017.csv", encoding='latin1')
ss2018 = pd.read_csv(f"{DATA_PATH}/SS2018.csv", encoding='latin1')
ss2019 = pd.read_csv(f"{DATA_PATH}/SS2019.csv", encoding='latin1')

# Standardize column names

ss2017_clean = ss2017[['State', 'Total Score\n2000']].copy()
ss2017_clean.columns = ['State', 'Score']
ss2017_clean['Year'] = 2017

ss2018_clean = ss2018[['Name of State/UT', 'Overall\nMarks\n(4000M)']].copy()
ss2018_clean.columns = ['State', 'Score']
ss2018_clean['Year'] = 2018

ss2019_clean = ss2019[['Name of the State/UT', 'Overall\nMarks (5000\nM)']].copy()
ss2019_clean.columns = ['State', 'Score']
ss2019_clean['Year'] = 2019

# Combine
sanitation_df = pd.concat(
    [ss2017_clean, ss2018_clean, ss2019_clean],
    ignore_index=True
)

print("Combined sanitation shape:", sanitation_df.shape)
sanitation_df.head()


**Normalize & Create State Sanitation Index**

In [ ]:
# =====================================================
# Cell 31: Normalize Sanitation Scores & Create Index
# =====================================================

# Normalize within each year (min-max scaling per year)
sanitation_df['NormalizedScore'] = sanitation_df.groupby('Year')['Score']\
    .transform(lambda x: (x - x.min()) / (x.max() - x.min()))

# Compute average sanitation index per state
state_sanitation_index = sanitation_df.groupby('State')[
    'NormalizedScore'
].mean().reset_index()

state_sanitation_index.rename(
    columns={'NormalizedScore': 'SanitationIndex'},
    inplace=True
)

print("State sanitation index created ✅")
state_sanitation_index.head()


**Clean State Names (Sanitation Data)**

In [ ]:
# =====================================================
# Cell 32: Clean State Names in Sanitation Index
# =====================================================

# Remove newline characters and strip spaces
state_sanitation_index['State'] = state_sanitation_index['State']\
    .str.replace('\n', ' ', regex=False)\
    .str.strip()

print("Cleaned state names ✅")
state_sanitation_index.head()


**Merge Sanitation Index with Outbreak Data**

In [ ]:
# =====================================================
# Cell 33: Merge Sanitation Index
# =====================================================

# Merge sanitation index
final_df = pd.merge(
    merged_df,
    state_sanitation_index,
    left_on='State/UnionTerritory',
    right_on='State',
    how='left'
)

# Drop duplicate state column
final_df.drop(columns=['State'], inplace=True)

# Fill missing sanitation values (if any) with mean
final_df['SanitationIndex'] = final_df['SanitationIndex']\
    .fillna(final_df['SanitationIndex'].mean())

print("Sanitation merged successfully ✅")
final_df.head()


**Sanitation Map**

In [ ]:
# =====================================================
# Sanitation Choropleth Map (Corrected)
# =====================================================

import folium

# Prepare sanitation dataframe
sanitation_geo = india_geo.merge(
    state_sanitation_index.rename(columns={'State': 'NAME_1'}),
    on='NAME_1',
    how='left'
)

# Fill missing values
sanitation_geo['SanitationIndex'] = \
    sanitation_geo['SanitationIndex'].fillna(0)

# Create base map
m_sanitation = folium.Map(location=[22.5, 80], zoom_start=5)

# Add choropleth
folium.Choropleth(
    geo_data=sanitation_geo,
    data=sanitation_geo,
    columns=['NAME_1', 'SanitationIndex'],
    key_on='feature.properties.NAME_1',
    fill_color='Greens',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Sanitation Index'
).add_to(m_sanitation)

m_sanitation


**Train Gradient Boosting with Sanitation + Vaccination**

In [ ]:
# =====================================================
# Cell 34: Train Full Model (Lag + Vaccine + Sanitation)
# =====================================================

# Time split
train = final_df[final_df['Year'] < 2021]
test = final_df[final_df['Year'] >= 2021]

feature_cols = [
    'lag_1',
    'lag_2',
    'rolling_mean_3',
    'MonthlyVaccinations',
    'SanitationIndex'
]

X_train = train[feature_cols]
y_train = train['MonthlyCases']

X_test = test[feature_cols]
y_test = test['MonthlyCases']

# Train model
gbr_full_model = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gbr_full_model.fit(X_train, y_train)

preds_full = gbr_full_model.predict(X_test)

# Evaluation
mae_full = mean_absolute_error(y_test, preds_full)
rmse_full = np.sqrt(mean_squared_error(y_test, preds_full))
r2_full = r2_score(y_test, preds_full)

print("Full Model Results")
print("MAE:", round(mae_full, 2))
print("RMSE:", round(rmse_full, 2))
print("R²:", round(r2_full, 3))


**Feature Importance Analysis**

In [ ]:
# =====================================================
# Cell 35: Feature Importance
# =====================================================

importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': gbr_full_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

importance_df


**Create Climate Trend Index**

In [ ]:
# =====================================================
# Cell 36: Create Climate Trend Index (Structural)
# =====================================================

# Load dataset
global_temp_df = pd.read_csv(
    f"{DATA_PATH}/GlobalTemperatures.csv",
    encoding='latin1'
)

# Convert date
global_temp_df['dt'] = pd.to_datetime(global_temp_df['dt'])

# Keep temperature column
climate_df = global_temp_df[['dt', 'LandAverageTemperature']].dropna()

# Use modern period
climate_df = climate_df[climate_df['dt'].dt.year >= 1950]

# Baseline period (1951–1980)
baseline = climate_df[
    (climate_df['dt'].dt.year >= 1951) &
    (climate_df['dt'].dt.year <= 1980)
]['LandAverageTemperature'].mean()

# Compute anomaly
climate_df['TempAnomaly'] = climate_df['LandAverageTemperature'] - baseline

# Compute structural climate index
climate_index = climate_df[
    (climate_df['dt'].dt.year >= 2011)
]['TempAnomaly'].mean()

print("Baseline Temperature:", round(baseline, 3))
print("Structural Climate Index (Recent Anomaly):", round(climate_index, 3))


**Add Climate Index to Final Dataset**

In [ ]:
# =====================================================
# Cell 37: Add Climate Index to Dataset
# =====================================================

final_df_climate = final_df.copy()

# Add structural climate index as constant column
final_df_climate['ClimateIndex'] = climate_index

print("Climate index added successfully ✅")
final_df_climate.head()


**Train Final Model (Lag + Vaccine + Sanitation + Climate)**

In [ ]:
# =====================================================
# Cell 38: Train Final Integrated Model
# =====================================================

# Time-based split
train = final_df_climate[final_df_climate['Year'] < 2021]
test = final_df_climate[final_df_climate['Year'] >= 2021]

feature_cols = [
    'lag_1',
    'lag_2',
    'rolling_mean_3',
    'MonthlyVaccinations',
    'SanitationIndex',
    'ClimateIndex'
]

X_train = train[feature_cols]
y_train = train['MonthlyCases']

X_test = test[feature_cols]
y_test = test['MonthlyCases']

# Train model
gbr_final_model = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gbr_final_model.fit(X_train, y_train)

preds_final = gbr_final_model.predict(X_test)

# Evaluation
mae_final = mean_absolute_error(y_test, preds_final)
rmse_final = np.sqrt(mean_squared_error(y_test, preds_final))
r2_final = r2_score(y_test, preds_final)

print("FINAL MODEL RESULTS")
print("MAE:", round(mae_final, 2))
print("RMSE:", round(rmse_final, 2))
print("R²:", round(r2_final, 3))


**Load & Inspect Testing Dataset**

In [ ]:
# =====================================================
# Cell 39: Load & Inspect Testing Dataset
# =====================================================

testing_df = pd.read_csv(f"{DATA_PATH}/StatewiseTestingDetails.csv")

print("Shape:", testing_df.shape)
print("\nColumns:\n", testing_df.columns)

testing_df.head()


**Convert Testing to Monthly New Tests**

In [ ]:
# =====================================================
# Cell 40: Convert Testing Data to Monthly
# =====================================================

# Convert date
testing_df['Date'] = pd.to_datetime(testing_df['Date'])

# Extract YearMonth
testing_df['YearMonth'] = testing_df['Date'].dt.to_period('M')

# Aggregate monthly (take max cumulative per month)
monthly_testing = testing_df.groupby(
    ['State', 'YearMonth']
)['TotalSamples'].max().reset_index()

# Convert YearMonth back to timestamp
monthly_testing['YearMonth'] = monthly_testing['YearMonth'].dt.to_timestamp()

# Sort
monthly_testing = monthly_testing.sort_values(
    by=['State', 'YearMonth']
)

# Convert cumulative to monthly new tests
monthly_testing['MonthlyTests'] = monthly_testing.groupby(
    'State'
)['TotalSamples'].diff()

monthly_testing['MonthlyTests'] = monthly_testing['MonthlyTests']\
    .fillna(monthly_testing['TotalSamples'])

monthly_testing['MonthlyTests'] = monthly_testing['MonthlyTests']\
    .apply(lambda x: 0 if x < 0 else x)

print("Monthly testing dataset ready ✅")
monthly_testing.head()


**Merge Testing into Final Dataset**

In [ ]:
# =====================================================
# Cell 41: Merge Testing Data
# =====================================================

# Rename for consistency
monthly_testing.rename(columns={'State': 'State/UnionTerritory'}, inplace=True)

# Merge with final climate dataset
final_df_all = pd.merge(
    final_df_climate,
    monthly_testing[['State/UnionTerritory', 'YearMonth', 'MonthlyTests']],
    on=['State/UnionTerritory', 'YearMonth'],
    how='left'
)

# Fill missing testing values with 0
final_df_all['MonthlyTests'] = final_df_all['MonthlyTests'].fillna(0)

print("Testing merged successfully ✅")
final_df_all.head()


**Testing Intensity Map**

In [ ]:
# =====================================================
# Testing Intensity Choropleth Map
# =====================================================

# Compute average monthly testing per state
testing_avg = final_df_all.groupby(
    'State/UnionTerritory'
)['MonthlyTests'].mean().reset_index()

# Merge with geo data
testing_geo = india_geo.merge(
    testing_avg.rename(columns={'State/UnionTerritory':'NAME_1'}),
    on='NAME_1',
    how='left'
)

# Fill missing
testing_geo['MonthlyTests'] = testing_geo['MonthlyTests'].fillna(0)

# Create base map
m_testing = folium.Map(location=[22.5, 80], zoom_start=5)

# Add choropleth
folium.Choropleth(
    geo_data=testing_geo,
    data=testing_geo,
    columns=['NAME_1', 'MonthlyTests'],
    key_on='feature.properties.NAME_1',
    fill_color='Blues',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Average Monthly Testing'
).add_to(m_testing)

m_testing


**Train Ultimate Final Model**

In [ ]:
# =====================================================
# Cell 42: Train Ultimate Final Model
# =====================================================

# Time-based split
train = final_df_all[final_df_all['Year'] < 2021]
test = final_df_all[final_df_all['Year'] >= 2021]

feature_cols = [
    'lag_1',
    'lag_2',
    'rolling_mean_3',
    'MonthlyVaccinations',
    'SanitationIndex',
    'ClimateIndex',
    'MonthlyTests'
]

X_train = train[feature_cols]
y_train = train['MonthlyCases']

X_test = test[feature_cols]
y_test = test['MonthlyCases']

# Train model
gbr_ultimate_model = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gbr_ultimate_model.fit(X_train, y_train)

preds_ultimate = gbr_ultimate_model.predict(X_test)

# Evaluation
mae_ultimate = mean_absolute_error(y_test, preds_ultimate)
rmse_ultimate = np.sqrt(mean_squared_error(y_test, preds_ultimate))
r2_ultimate = r2_score(y_test, preds_ultimate)

print("ULTIMATE FINAL MODEL RESULTS")
print("MAE:", round(mae_ultimate, 2))
print("RMSE:", round(rmse_ultimate, 2))
print("R²:", round(r2_ultimate, 3))


**Mathematical Model Definition**

In [ ]:
# =====================================================
# Cell 42A: Formal Mathematical Model Definition
# =====================================================

print("Mathematical Formulation:\n")

print("MonthlyCases(s,t) = f(")
print("lag_1, lag_2, rolling_mean_3,")
print("MonthlyVaccinations, MonthlyTests,")
print("SanitationIndex, ClimateIndex)")
print("\nWhere:")
print("s = state")
print("t = month")

print("\nVulnerability Index Formula:")
print("V_s = (NormCases + (1 - NormSanitation) + (1 - NormTesting)) / 3")


**TimeSeries Cross Validation**

In [ ]:
# =====================================================
# Cell 42B: TimeSeries Cross-Validation (Robustness Test)
# =====================================================

from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

X_all = final_df_all[feature_cols]
y_all = final_df_all['MonthlyCases']

cv_mae = []
cv_rmse = []
cv_r2 = []

for train_index, test_index in tscv.split(X_all):

    X_train_cv, X_test_cv = X_all.iloc[train_index], X_all.iloc[test_index]
    y_train_cv, y_test_cv = y_all.iloc[train_index], y_all.iloc[test_index]

    model_cv = GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )

    model_cv.fit(X_train_cv, y_train_cv)
    preds_cv = model_cv.predict(X_test_cv)

    cv_mae.append(mean_absolute_error(y_test_cv, preds_cv))
    cv_rmse.append(np.sqrt(mean_squared_error(y_test_cv, preds_cv)))
    cv_r2.append(r2_score(y_test_cv, preds_cv))

print("TimeSeries CV Results:")
print("Average MAE:", round(np.mean(cv_mae), 2))
print("Average RMSE:", round(np.mean(cv_rmse), 2))
print("Average R²:", round(np.mean(cv_r2), 3))


**Final Feature Importance**

In [ ]:
# =====================================================
# Cell 43: Ultimate Model Feature Importance
# =====================================================

importance_df_final = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': gbr_ultimate_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

importance_df_final


# **SHAP Analysis**

In [ ]:
# =====================================================
# Cell 43A: SHAP Explainability
# =====================================================

import sys
!{sys.executable} -m pip install shap -q

import shap

explainer = shap.TreeExplainer(gbr_ultimate_model)
shap_values = explainer.shap_values(X_test)

print("Generating SHAP Summary Plot...")
shap.summary_plot(shap_values, X_test)


**Feature Importance Bar Chart**

In [ ]:
# =====================================================
# Feature Importance Visualization
# =====================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.barh(
    importance_df_final['Feature'],
    importance_df_final['Importance']
)

plt.xlabel("Importance Score")
plt.title("Ultimate Model Feature Importance")
plt.gca().invert_yaxis()

plt.show()


**Compute State-Level Aggregates**

In [ ]:
# =====================================================
# Cell 44: Compute State-Level Aggregates
# =====================================================

# Add predictions to test set
test['PredictedCases'] = preds_ultimate

# Aggregate per state
state_agg = test.groupby('State/UnionTerritory').agg({
    'PredictedCases': 'mean',
    'SanitationIndex': 'mean',
    'MonthlyTests': 'mean'
}).reset_index()

state_agg.head()


**Normalize & Compute Vulnerability **Index

In [ ]:
# =====================================================
# Cell 45: Compute Vulnerability Index
# =====================================================

# Copy
vuln_df = state_agg.copy()

# Normalize function
def minmax(series):
    return (series - series.min()) / (series.max() - series.min())

# Normalize components
vuln_df['NormCases'] = minmax(vuln_df['PredictedCases'])
vuln_df['NormSanitation'] = minmax(vuln_df['SanitationIndex'])
vuln_df['NormTesting'] = minmax(vuln_df['MonthlyTests'])

# Invert sanitation & testing (low value = high vulnerability)
vuln_df['SanitationRisk'] = 1 - vuln_df['NormSanitation']
vuln_df['TestingRisk'] = 1 - vuln_df['NormTesting']

# Compute composite vulnerability score
vuln_df['VulnerabilityIndex'] = (
    vuln_df['NormCases'] +
    vuln_df['SanitationRisk'] +
    vuln_df['TestingRisk']
) / 3

# Sort highest risk first
vuln_df = vuln_df.sort_values(by='VulnerabilityIndex', ascending=False)

vuln_df[['State/UnionTerritory', 'VulnerabilityIndex']].head(10)


**Risk Categories**

In [ ]:
# =====================================================
# Risk Classification (Low / Medium / High)
# =====================================================

# Create 3 quantile-based categories
vuln_df['RiskLevel'] = pd.qcut(
    vuln_df['VulnerabilityIndex'],
    q=3,
    labels=['Low Risk', 'Medium Risk', 'High Risk']
)

vuln_df[['State/UnionTerritory', 'VulnerabilityIndex', 'RiskLevel']].head()


**Merging Risk Levels into Geo Data**

In [ ]:
# =====================================================
# Merge Risk Classification with Geo Data
# =====================================================

# Prepare vulnerability dataframe
vuln_geo = vuln_df.rename(columns={
    'State/UnionTerritory': 'NAME_1'
})

# Apply earlier name corrections
name_corrections = {
    "Andaman and Nicobar Islands": "Andaman and Nicobar",
    "Jammu & Kashmir": "Jammu and Kashmir",
    "NCT of Delhi": "Delhi"
}

vuln_geo['NAME_1'] = vuln_geo['NAME_1'].replace(name_corrections)

# Merge
india_risk_map = india_geo.merge(
    vuln_geo[['NAME_1', 'VulnerabilityIndex', 'RiskLevel']],
    on='NAME_1',
    how='left'
)

# Fill missing
india_risk_map['VulnerabilityIndex'] = \
    india_risk_map['VulnerabilityIndex'].fillna(0)

india_risk_map['RiskLevel'] = \
    india_risk_map['RiskLevel'].fillna('Low Risk')

india_risk_map.head()


**Classified Risk Map**

In [ ]:
# =====================================================
# Classified Risk Map (Low / Medium / High)
# =====================================================

import folium

# Create base map
m_classified = folium.Map(location=[22.5, 80], zoom_start=5)

# Define color mapping
risk_colors = {
    'Low Risk': '#2ecc71',      # Green
    'Medium Risk': '#f1c40f',   # Yellow
    'High Risk': '#e74c3c'      # Red
}

# Add colored GeoJson layer
folium.GeoJson(
    india_risk_map,
    style_function=lambda feature: {
        'fillColor': risk_colors.get(
            feature['properties']['RiskLevel'], '#2ecc71'
        ),
        'color': 'black',
        'weight': 0.5,
        'fillOpacity': 0.7
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['NAME_1', 'VulnerabilityIndex', 'RiskLevel'],
        aliases=['State:', 'Vulnerability Score:', 'Risk Level:'],
        localize=True
    )
).add_to(m_classified)

m_classified


**Merge Vulnerability with Geo Data**

In [ ]:
# =====================================================
# Cell 48: Merge Vulnerability with Geo Data
# =====================================================

# Rename columns for merging
vuln_geo = vuln_df.rename(columns={
    'State/UnionTerritory': 'NAME_1'
})

# Merge
india_merged = india_geo.merge(
    vuln_geo,
    on='NAME_1',
    how='left'
)

print("Merge complete ✅")
india_merged[['NAME_1', 'VulnerabilityIndex']].head()


**Fix State Name Differences**

In [ ]:
# =====================================================
# Cell 49: Fix State Name Mapping
# =====================================================

# Create copy
vuln_geo = vuln_df.copy()

# Manual corrections
name_corrections = {
    "Andaman and Nicobar Islands": "Andaman and Nicobar",
    "Jammu & Kashmir": "Jammu and Kashmir",
    "NCT of Delhi": "Delhi"
}

vuln_geo['NAME_1'] = vuln_geo['State/UnionTerritory']\
    .replace(name_corrections)

# Merge again
india_merged = india_geo.merge(
    vuln_geo,
    on='NAME_1',
    how='left'
)

print("Merge corrected ✅")

india_merged[['NAME_1', 'VulnerabilityIndex']].head(10)


**Fill Missing Vulnerability Values**

In [ ]:
# =====================================================
# Cell 50: Fill Missing Vulnerability Values
# =====================================================

india_merged['VulnerabilityIndex'] = \
    india_merged['VulnerabilityIndex'].fillna(0)

print("Remaining NaN values:",
      india_merged['VulnerabilityIndex'].isna().sum())


**Create Choropleth Vulnerability Map**

In [ ]:
# =====================================================
# Cell 51: Generate Vulnerability Choropleth Map
# =====================================================

import folium

# Create base map centered on India
m = folium.Map(location=[22.5, 80], zoom_start=5)

# Add choropleth layer
folium.Choropleth(
    geo_data=india_merged,
    data=india_merged,
    columns=['NAME_1', 'VulnerabilityIndex'],
    key_on='feature.properties.NAME_1',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Outbreak Vulnerability Index'
).add_to(m)

# Add hover tooltip
folium.GeoJson(
    india_merged,
    tooltip=folium.features.GeoJsonTooltip(
        fields=['NAME_1', 'VulnerabilityIndex'],
        aliases=['State:', 'Vulnerability Score:'],
        localize=True
    )
).add_to(m)

m


**Moran’s I Test**

In [ ]:
# =====================================================
# Cell 52: Spatial Autocorrelation (Moran's I)
# =====================================================

!{sys.executable} -m pip install esda libpysal -q


import libpysal
from esda.moran import Moran

# Prepare GeoDataFrame
geo_moran = india_merged.copy()

# Build spatial weights
w = libpysal.weights.Queen.from_dataframe(geo_moran)
w.transform = 'r'

# Moran's I calculation
moran = Moran(geo_moran['VulnerabilityIndex'], w)

print("Moran's I:", round(moran.I, 4))
print("p-value:", round(moran.p_sim, 4))


MODEL COMPARISON TABLE

In [ ]:
# =====================================================
# FINAL CELL — MODEL COMPARISON + IEEE STYLE GRAPH
# =====================================================

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------------------------------
# CREATE FINAL COMPARISON TABLE
# -----------------------------------------------------
comparison_final = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "Poisson Regression",
        "Gradient Boosting",
        "Ultimate Model"
    ],
    "MAE": [
        results["Linear Regression"]["MAE"],
        results["Random Forest"]["MAE"],
        poisson_mae,
        gbr_mae,
        mae_ultimate if "mae_ultimate" in globals() else np.nan
    ],
    "RMSE": [
        results["Linear Regression"]["RMSE"],
        results["Random Forest"]["RMSE"],
        poisson_rmse,
        gbr_rmse,
        rmse_ultimate if "rmse_ultimate" in globals() else np.nan
    ],
    "R2": [
        results["Linear Regression"]["R2"],
        results["Random Forest"]["R2"],
        poisson_r2,
        gbr_r2,
        r2_ultimate if "r2_ultimate" in globals() else np.nan
    ]
})

print("\nFINAL MODEL COMPARISON TABLE\n")
display(comparison_final)


# -----------------------------------------------------
# IEEE-STYLE VISUALIZATION
# (Bars = Errors, Line = R²)
# -----------------------------------------------------
fig, ax1 = plt.subplots(figsize=(10,5))

# Error bars
comparison_final.set_index("Model")[["MAE","RMSE"]].plot(
    kind="bar",
    ax=ax1
)

ax1.set_ylabel("Error Metrics")

# R2 line
ax2 = ax1.twinx()
comparison_final.set_index("Model")["R2"].plot(
    marker="o",
    ax=ax2
)

ax2.set_ylabel("R² Score")

plt.title("Model Performance Comparison")
plt.xticks(rotation=45)
plt.tight_layout()

# Save figure (for paper)
plt.savefig("model_comparison.png", dpi=300, bbox_inches="tight")

plt.show()


# -----------------------------------------------------
# SAVE TABLE FOR PAPER
# -----------------------------------------------------
comparison_final.to_csv("model_comparison_results.csv", index=False)

print("\nSaved files:")
print("✔ model_comparison.png")
print("✔ model_comparison_results.csv")

In [ ]:
# SECTION 1: Create Dummy Dataset for Tamil Nadu (38 Districts)

import pandas as pd
import numpy as np

np.random.seed(42)

districts = [
    "Ariyalur", "Chengalpattu", "Chennai", "Coimbatore", "Cuddalore",
    "Dharmapuri", "Dindigul", "Erode", "Kallakurichi", "Kancheepuram",
    "Kanniyakumari", "Karur", "Krishnagiri", "Madurai", "Mayiladuthurai",
    "Nagapattinam", "Namakkal", "Nilgiris", "Perambalur", "Pudukkottai",
    "Ramanathapuram", "Ranipet", "Salem", "Sivagangai", "Tenkasi",
    "Thanjavur", "Theni", "Thoothukudi", "Tiruchirappalli", "Tirunelveli",
    "Tirupathur", "Tiruppur", "Tiruvallur", "Tiruvannamalai",
    "Tiruvarur", "Vellore", "Viluppuram", "Virudhunagar"
]

df = pd.DataFrame({
    "District": districts,
    "PredictedCases": np.random.randint(500, 5000, size=38),
    "SanitationIndex": np.round(np.random.uniform(0.45, 0.95, size=38), 3),
    "MonthlyTests": np.random.randint(10000, 120000, size=38)
})

# Normalize function
def minmax(series):
    return (series - series.min()) / (series.max() - series.min())

df["NormCases"] = minmax(df["PredictedCases"])
df["NormSanitation"] = minmax(df["SanitationIndex"])
df["NormTesting"] = minmax(df["MonthlyTests"])

df["SanitationRisk"] = 1 - df["NormSanitation"]
df["TestingRisk"] = 1 - df["NormTesting"]

df["VulnerabilityIndex"] = (
    df["NormCases"] +
    df["SanitationRisk"] +
    df["TestingRisk"]
) / 3

df["RiskLevel"] = pd.qcut(
    df["VulnerabilityIndex"],
    q=3,
    labels=["Low Risk", "Medium Risk", "High Risk"]
)

df = df.sort_values("VulnerabilityIndex", ascending=False)

# Show top rows
df.head()

In [ ]:
# SECTION 2: Save Dataset

df.to_csv("tn_district_dummy_data.csv", index=False)

print("File saved successfully!")

# Optional: verify
df.tail()

In [ ]:
# =====================================================
# Check GeoJSON File
# =====================================================

import geopandas as gpd

# Replace with your actual file name if different
geo = gpd.read_file("/content/data/TamilNadu.geojson")

print("✅ File loaded successfully")

print("\n📊 Shape:")
print(geo.shape)

print("\n📌 Columns:")
print(list(geo.columns))

print("\n👀 Preview:")
geo.head()

In [ ]:
# =====================================================
# Fix District Name Differences
# =====================================================

name_corrections = {
    "Tiruchirappalli": "Tiruchchirappalli",
    "Thoothukudi": "Thoothukudi",
    "Tirunelveli": "Tirunelveli Kattabo",  # dataset uses extended name
}

df["District"] = df["District"].replace(name_corrections)

In [ ]:
# =====================================================
# Merge Data with Real GIS
# =====================================================

tn_map_df = geo.merge(
    df,
    left_on="NAME_2",
    right_on="District",
    how="left"
)

print("Merge done ✅")

tn_map_df[["NAME_2", "VulnerabilityIndex"]].head()

In [ ]:
# =====================================================
# Convert Vulnerability Index to Percentage
# =====================================================

tn_map_df["VulnerabilityPercent"] = (tn_map_df["VulnerabilityIndex"] * 100).round(2)


# =====================================================
# Standard Risk Classification (ADD THIS HERE)
# =====================================================

def classify_risk(v):
    if v < 33:
        return "Low Risk"
    elif v < 66:
        return "Medium Risk"
    else:
        return "High Risk"

tn_map_df["RiskLevel"] = tn_map_df["VulnerabilityPercent"].apply(classify_risk)

In [ ]:
# =====================================================
# FINAL District-Level GIS Map (Percentage View)
# =====================================================

import folium

m = folium.Map(location=[11, 78], zoom_start=7)

folium.Choropleth(
    geo_data=tn_map_df,
    data=tn_map_df,
    columns=["District", "VulnerabilityPercent"],   # UPDATED
    key_on="feature.properties.NAME_2",
    fill_color="YlOrRd",
    fill_opacity=0.7,
    line_opacity=0.5,
    legend_name="Vulnerability (%)"   # UPDATED
).add_to(m)

# Tooltip with percentage formatting
folium.GeoJson(
    tn_map_df,
    tooltip=folium.GeoJsonTooltip(
        fields=["NAME_2", "VulnerabilityPercent"],
        aliases=["District:", "Vulnerability (%):"],
        localize=True
    )
).add_to(m)

m

In [ ]:
# Check actual ranges

tn_map_df.groupby("RiskLevel")["VulnerabilityPercent"].agg(["min", "max"])